# 2WikiMultiHopQA Data Exploration

This notebook:
1. Validates dataset loading and inspects corpus statistics
2. Analyzes question type distribution
3. **Checks demographic distribution** (gender, geography) to validate fairness evaluation feasibility
4. Examines evidence triple coverage and entity_ids availability

In [ ]:
from pathlib import Path
import sys
from collections import Counter

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

from fair_kg_rag.data.dataset_loader import load_dataset
from fair_kg_rag.utils.io_utils import read_json

plt.rcParams['figure.dpi'] = 100
sns.set_style('whitegrid')

## 1. Load Dataset

In [ ]:
raw_path = ROOT / 'data' / 'raw' / 'dev.json'
if not raw_path.exists():
    raise FileNotFoundError(
        f'Missing {raw_path}. Run: python scripts/download_data.py'
    )

records = load_dataset(raw_path)
print(f'Loaded {len(records)} records from dev split')

## 2. Corpus Statistics

In [ ]:
rows = []
for record in records:
    rows.append({
        'id': record.id,
        'question_type': record.question_type,
        'question_length': len(record.question.split()),
        'num_paragraphs': len(record.context),
        'num_sentences': sum(len(p.sentences) for p in record.context),
        'num_supporting_facts': len(record.supporting_facts),
        'num_evidence_triples': len(record.evidences),
        'num_entity_ids': len(record.wikidata_ids),
        'has_evidences': len(record.evidences) > 0,
        'has_entity_ids': len(record.wikidata_ids) > 0,
        'entity_ids': record.entity_ids,
    })

df = pd.DataFrame(rows)

print('=== Dataset Summary ===')
print(f"Total questions: {len(df)}")
print(f"Avg question length: {df['question_length'].mean():.1f} words")
print(f"Avg paragraphs/question: {df['num_paragraphs'].mean():.1f}")
print(f"Avg sentences/question: {df['num_sentences'].mean():.1f}")
print(f"Avg evidence triples/question: {df['num_evidence_triples'].mean():.1f}")
print(f"\nQuestions with evidence triples: {df['has_evidences'].sum()} / {len(df)} ({df['has_evidences'].mean()*100:.1f}%)")
print(f"Questions with entity_ids: {df['has_entity_ids'].sum()} / {len(df)} ({df['has_entity_ids'].mean()*100:.1f}%)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Question type distribution
type_counts = df['question_type'].value_counts()
type_counts.plot.bar(ax=axes[0], color='steelblue')
axes[0].set_title('Question Type Distribution')
axes[0].set_xlabel('')
axes[0].set_ylabel('Count')
for i, v in enumerate(type_counts.values):
    axes[0].text(i, v + 20, str(v), ha='center', fontsize=9)

# Evidence triple coverage by type
type_evidence = df.groupby('question_type')['has_evidences'].mean() * 100
type_evidence.plot.bar(ax=axes[1], color='darkorange')
axes[1].set_title('% Questions with Evidence Triples (by type)')
axes[1].set_xlabel('')
axes[1].set_ylabel('Percentage')
axes[1].set_ylim(0, 105)

plt.tight_layout()
plt.show()

## 3. Demographic Distribution Analysis

**Critical check**: Are demographics balanced enough for meaningful fairness evaluation?

We use `entity_ids` (Wikidata QIDs) to fetch gender (P21) and nationality (P27) from Wikidata.
If the dataset is heavily skewed (e.g., 95% male), fairness metrics are less informative.

In [ ]:
# Load demographics (must have run: python scripts/fetch_demographics.py)
demo_path = ROOT / 'data' / 'processed' / 'entity_demographics.json'

if demo_path.exists():
    demographics = read_json(demo_path)
    demo_df = pd.DataFrame(demographics)
    print(f'Loaded demographics for {len(demo_df)} entities')
    print(f'\nColumns: {list(demo_df.columns)}')
    demo_df.head()
else:
    print(f'Demographics not found at {demo_path}')
    print('Run: python scripts/fetch_demographics.py --split dev')
    demo_df = None

In [ ]:
if demo_df is not None:
    print('=== Gender Distribution ===')
    gender_counts = demo_df['gender'].value_counts(dropna=False)
    print(gender_counts)
    print(f'\nNull/missing: {demo_df["gender"].isna().sum()} ({demo_df["gender"].isna().mean()*100:.1f}%)')
    
    print('\n=== Geographic Group Distribution ===')
    geo_counts = demo_df['geo_group'].value_counts(dropna=False)
    print(geo_counts)
    print(f'\nNull/missing: {(demo_df["geo_group"] == "unknown").sum()} unknown')

In [ ]:
if demo_df is not None:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    # Gender distribution (exclude None)
    gender_valid = demo_df[demo_df['gender'].notna()]['gender'].value_counts()
    if len(gender_valid) > 0:
        gender_valid.plot.pie(ax=axes[0], autopct='%1.1f%%', startangle=90)
        axes[0].set_title(f'Gender Distribution (n={gender_valid.sum()})')
        axes[0].set_ylabel('')
    else:
        axes[0].text(0.5, 0.5, 'No gender data', ha='center', va='center')
    
    # Geo group distribution (exclude unknown)
    geo_valid = demo_df[demo_df['geo_group'] != 'unknown']['geo_group'].value_counts()
    if len(geo_valid) > 0:
        geo_valid.plot.pie(ax=axes[1], autopct='%1.1f%%', startangle=90)
        axes[1].set_title(f'Geographic Group Distribution (n={geo_valid.sum()})')
        axes[1].set_ylabel('')
    else:
        axes[1].text(0.5, 0.5, 'No geo data', ha='center', va='center')
    
    plt.tight_layout()
    plt.show()

## 4. Per-Question Demographic Assignment

Map demographics to questions via `entity_ids` field.
Each question has 2 (or 4 for bridge_comparison) Wikidata entity IDs.

In [ ]:
if demo_df is not None:
    demo_by_qid = {row['qid']: row for _, row in demo_df.iterrows()}
    
    # Assign demographics to each question
    question_genders = []
    question_geos = []
    
    for record in records:
        gender = None
        geo = None
        for qid in record.wikidata_ids:
            d = demo_by_qid.get(qid, {})
            if isinstance(d, pd.Series):
                d = d.to_dict()
            if d.get('gender') and gender is None:
                gender = d['gender']
            if d.get('geo_group') and d.get('geo_group') != 'unknown' and geo is None:
                geo = d['geo_group']
        question_genders.append(gender)
        question_geos.append(geo)
    
    df['gender'] = question_genders
    df['geo_group'] = question_geos
    
    print('=== Questions by Gender ===')
    print(df['gender'].value_counts(dropna=False))
    print(f'\nCoverage: {df["gender"].notna().sum()} / {len(df)} ({df["gender"].notna().mean()*100:.1f}%)')
    
    print('\n=== Questions by Geographic Group ===')
    print(df['geo_group'].value_counts(dropna=False))
    print(f'\nCoverage: {df["geo_group"].notna().sum()} / {len(df)} ({df["geo_group"].notna().mean()*100:.1f}%)')

In [ ]:
if demo_df is not None and 'gender' in df.columns:
    # Cross-tabulation: question_type x gender
    ct = pd.crosstab(df['question_type'], df['gender'], margins=True)
    print('=== Question Type x Gender Cross-Tab ===')
    print(ct)
    print()
    
    # Cross-tabulation: question_type x geo_group
    ct_geo = pd.crosstab(df['question_type'], df['geo_group'], margins=True)
    print('=== Question Type x Geographic Group Cross-Tab ===')
    print(ct_geo)

In [ ]:
if demo_df is not None and 'gender' in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Gender by question type
    gender_by_type = df[df['gender'].notna()].groupby(['question_type', 'gender']).size().unstack(fill_value=0)
    if not gender_by_type.empty:
        gender_by_type.plot.bar(ax=axes[0], stacked=True)
        axes[0].set_title('Gender Distribution by Question Type')
        axes[0].set_xlabel('')
        axes[0].legend(title='Gender')
    
    # Geo by question type
    geo_by_type = df[df['geo_group'].notna()].groupby(['question_type', 'geo_group']).size().unstack(fill_value=0)
    if not geo_by_type.empty:
        geo_by_type.plot.bar(ax=axes[1], stacked=True)
        axes[1].set_title('Geographic Group by Question Type')
        axes[1].set_xlabel('')
        axes[1].legend(title='Geo Group')
    
    plt.tight_layout()
    plt.show()

## 5. Fairness Evaluation Feasibility

For fairness metrics to be meaningful we need:
- At least 2 demographic groups with sufficient samples
- Neither group should dominate >90% of the data
- Enough coverage (questions that can be assigned a demographic label)

In [ ]:
if demo_df is not None and 'gender' in df.columns:
    print('=== FAIRNESS EVALUATION FEASIBILITY ===')
    print()
    
    # Gender check
    gender_valid = df[df['gender'].notna()]
    if len(gender_valid) > 0:
        gender_dist = gender_valid['gender'].value_counts(normalize=True)
        max_ratio = gender_dist.iloc[0]
        print(f'Gender coverage: {len(gender_valid)}/{len(df)} questions ({len(gender_valid)/len(df)*100:.1f}%)')
        print(f'Dominant group: {gender_dist.index[0]} ({max_ratio*100:.1f}%)')
        if max_ratio > 0.90:
            print('WARNING: Gender too skewed (>90%) — fairness metrics may not be informative')
        elif max_ratio > 0.70:
            print('NOTE: Moderate gender imbalance — fairness metrics applicable but interpret with caution')
        else:
            print('GOOD: Gender distribution balanced enough for meaningful fairness evaluation')
    else:
        print('WARNING: No gender data available')
    
    print()
    
    # Geo check
    geo_valid = df[df['geo_group'].notna()]
    if len(geo_valid) > 0:
        geo_dist = geo_valid['geo_group'].value_counts(normalize=True)
        max_ratio = geo_dist.iloc[0]
        print(f'Geo coverage: {len(geo_valid)}/{len(df)} questions ({len(geo_valid)/len(df)*100:.1f}%)')
        print(f'Dominant group: {geo_dist.index[0]} ({max_ratio*100:.1f}%)')
        if max_ratio > 0.90:
            print('WARNING: Geography too skewed (>90%) — fairness metrics may not be informative')
        elif max_ratio > 0.70:
            print('NOTE: Moderate geographic imbalance — interpret fairness results with caution')
        else:
            print('GOOD: Geographic distribution balanced for meaningful fairness evaluation')
    else:
        print('WARNING: No geographic data available')
    
    print()
    print('--- Conclusion ---')
    total_coverage = max(len(gender_valid), len(geo_valid))
    if total_coverage / len(df) < 0.10:
        print('INSUFFICIENT demographic coverage. Fairness evaluation not meaningful.')
    else:
        print(f'Sufficient demographic coverage ({total_coverage/len(df)*100:.0f}%) for fairness evaluation.')

## 6. Sample Records Inspection

In [ ]:
# Show a sample from each question type
for qtype in df['question_type'].unique():
    sample = next(r for r in records if r.question_type == qtype)
    print(f'\n--- {qtype.upper()} ---')
    print(f'Q: {sample.question}')
    print(f'A: {sample.answer}')
    print(f'Entity IDs: {sample.entity_ids}')
    print(f'Evidence triples: {len(sample.evidences)}')
    if sample.evidences:
        ev = sample.evidences[0]
        print(f'  Example: ({ev.subject}, {ev.relation}, {ev.obj})')